# Preparations

Compare IO-author classification within a campaign and on a completely held-out campaign. The transductive section also summarizes saved Step 3 classification sweeps and indicator information gain. Cross-campaign evaluation uses leave-one-campaign-out only.

Feature loading and per-campaign normalization retain the existing methodology, including normalization over each complete campaign.

## Imports

In [ ]:
from __future__ import annotations

# File system
from pathlib import Path
import json
import time

# Data
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# sklearn and models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedShuffleSplit,
    cross_validate,
)
from xgboost import XGBClassifier

# Utilities
from tqdm.auto import tqdm
from matplotlib.lines import Line2D
from IPython.display import display

## Parameters

Papermill injects overrides into the tagged cell below. Training fractions and ROC_TRAIN_FRACTION apply only to within-campaign evaluation.

In [ ]:
# Campaigns - each must have a completed step-3 sweep run at the standard location
CAMPAIGNS = ["Iran_1", "UAE", "Venezuela", "China_1", "Cuba", "Russia_1"]

output_dir = "results/Labeled_Datasets/step_4_inter_intra_classification"

# Sweep configuration - used to build each campaign's exact indicator file path
graph_filtering_setting = "all_authors"  # "all_authors" or "graph_authors"
topic_method = "BERTopic_topic_256"  # BERTopic_topic_128, K_Means_topic_124
edge_types = "similarity+text_similarity+interactions"
top_percentage = 100

# Fractions of labeled target-campaign accounts used for training.
FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 0.80]
ROC_TRAIN_FRACTION = 0.01
N_SPLITS = 5

MODELS = ["xgboost", "random_forest"] # "logistic_regression"
RANDOM_STATE = 42
indicators_configuration = "All"

In [ ]:
def validate_percentage(value: float | int, value_name: str) -> float:
    """Return a percentage after validating the shared 1-100 scale.

    Args:
        value: Percentage value from 1 through 100.
        value_name: Parameter name used in an error message.

    Returns:
        Validated percentage as a float.

    Raises:
        ValueError: If value is outside the inclusive range [1, 100].
    """
    percentage = float(value)
    if not 1 <= percentage <= 100:
        raise ValueError(
            f"{value_name} must be in [1, 100], where 1 means 1%; "
            f"got {value}."
        )
    return percentage

top_percentage = validate_percentage(top_percentage, "top_percentage")

## Paths and reporting

In [ ]:
def find_project_root(start_path):
    """Find the Backbone-Graph project root containing AGENTS.md.

    Args:
        start_path: Directory from which to begin the search.

    Returns:
        Path to the Backbone-Graph project root.
    """
    start_path = Path(start_path).resolve()
    for directory in (start_path, *start_path.parents):
        if (directory / "AGENTS.md").exists():
            return directory
    raise FileNotFoundError("Could not find the Backbone-Graph project root.")

project_root = find_project_root(Path.cwd())
results_root = project_root / "results" / "Labeled_Datasets"
CONFIG_TAG = (
    f"{graph_filtering_setting}_{topic_method}_{edge_types}"
    f"_top_percentage_{top_percentage:g}"
)
output_root = Path(output_dir).expanduser()
if not output_root.is_absolute():
    output_root = project_root / output_root
campaign_group_name = "_".join(sorted(campaign.lower().replace(" ", "_") for campaign in CAMPAIGNS))
transductive_output_path = output_root / "transductive_within_campaign" / campaign_group_name
inductive_output_path = output_root / "inductive_cross_campaign" / campaign_group_name
summary_output_path = transductive_output_path / f"aggregated_top_percentage_{top_percentage:g}"
for folder in (transductive_output_path, inductive_output_path, summary_output_path):
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def write_report(msg: str, output_path: Path | None = None) -> None:
    """Print progress and optionally append it to a section's report.

    Args:
        msg: Message to report.
        output_path: Section output directory, or None for display only.
    """
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}"
    print(line)
    if output_path is not None:
        with (output_path / "progress_report.txt").open("a", encoding="utf-8") as report_file:
            report_file.write(line + "\n")


def save_summary(table: pd.DataFrame, output_path: Path, name: str) -> None:
    """Export a summary as CSV and LaTeX.

    Args:
        table: Summary table to export without its index.
        output_path: Destination directory.
        name: Filename without an extension.
    """
    output_path.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_path / f"{name}.csv", index=False)
    table.to_latex(output_path / f"{name}.tex", index=False, float_format="%.3f", escape=True)

In [ ]:
def build_indicator_path(campaign: str, graph_filtering_setting: str) -> Path:
    """Build a campaign's step-3 indicator file path from the sweep config.

    Args:
        campaign: Campaign name (matches its results directory name).
        graph_filtering_setting: Graph filtering setting (e.g., "all_authors" or "graph_authors").

    Returns:
        Path to the expected `classification_sweep_indicators` parquet file.
    """
    base_dir = (
        results_root / campaign / f"{campaign}_part_all"
        / "step_3_key_authors_graph" / "classification_sweep"
        / "classification_sweep_indicators" / graph_filtering_setting
    )
    sweep_filename = (
        f"accounts_graph_{topic_method}_{edge_types}"
        f"_top_percentage_{top_percentage:g}_per_topic"
        f"_topic_col_{topic_method}_undirected_indicators.parquet"
    )
    return base_dir / sweep_filename


campaign_base_dirs = {campaign: build_indicator_path(campaign, graph_filtering_setting) for campaign in CAMPAIGNS}

print("Campaigns' paths:")
for campaign, path in campaign_base_dirs.items():
    print(f"  {campaign}: {path}")


for campaign, path in campaign_base_dirs.items():
    if not path.exists():
        raise FileNotFoundError(
            f"No data found for '{campaign}' at: {path}\n"
            f"Has the pipeline completed step 3 for this campaign?"
        )

## Data loading and classifiers

Labels come from `io_drive_label`, with the existing Step 3 JSON fallback. Shared numeric features exclude labels and campaign identifiers.

In [ ]:
def get_classifier(model_name: str, random_state: int = RANDOM_STATE, **kwargs):
    """Return a sklearn-compatible classifier by name.

    Args:
        model_name: One of "xgboost"/"xgb", "random_forest"/"rf",
            "logistic_regression"/"lr".
        random_state: Random seed forwarded to the classifier constructor.
        **kwargs: Additional keyword arguments passed to the constructor.

    Returns:
        An untrained sklearn-compatible classifier instance.

    Raises:
        ValueError: If model_name is not recognised.
    """
    name = model_name.lower()
    if name in ("xgboost", "xgb"):
        return XGBClassifier(
            random_state=random_state,
            eval_metric="logloss",
            n_jobs=1,
            **kwargs,
        )
    if name in ("random_forest", "rf"):
        return RandomForestClassifier(
            random_state=random_state, n_jobs=1, **kwargs
        )
    if name in ("logistic_regression", "lr"):
        return LogisticRegression(
            random_state=random_state,
            solver="liblinear",
            max_iter=1000,
            n_jobs=1,
            **kwargs,
        )
    raise ValueError(
        f"Unsupported model_name: '{model_name}'. "
        "Supported: xgboost/xgb, random_forest/rf, logistic_regression/lr."
    )

In [ ]:
def load_country_data(
    campaign_name: str,
    indicator_path: str | Path,
    prefix_ids: bool = True,
    normalize: bool = True,
    feature_columns: list[str] | None = None,
) -> tuple[pd.DataFrame, set[str]]:
    """Load one campaign's node indicators and IO-author labels.

    Args:
        campaign_name: Campaign label added to every returned row.
        indicator_path: Path to the campaign indicator parquet file.
        prefix_ids: Whether to prefix author IDs with the campaign name.
        normalize: Whether to min-max normalize numeric features within the
            complete campaign. Constant columns are set to zero.
        feature_columns: Optional ordered numeric feature list.

    Returns:
        A feature DataFrame with a ``campaign_name`` column and the set of IO
        author IDs using the same optional campaign prefix.

    Raises:
        FileNotFoundError: If the indicator file does not exist.
        ValueError: If labels or requested features are unavailable.
    """
    campaign_name = campaign_name.strip()
    path = Path(indicator_path).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"No indicator file for '{campaign_name}' at: {path}")

    df_features = pd.read_parquet(path)
    if "io_drive_label" in df_features.columns:
        io_labels = pd.to_numeric(df_features["io_drive_label"], errors="coerce")
        io_set = set(df_features.index[io_labels.eq(1)].astype(str))
    else:
        step_3_dir = path.parents[3]
        label_paths = sorted(
            step_3_dir.glob(
                "topic_col_*/io_author_analysis/io_authors_by_post_ratio.json"
            )
        )
        if not label_paths:
            raise ValueError(
                f"'{path}' has no 'io_drive_label' column and no "
                "io_author_analysis/io_authors_by_post_ratio.json was found "
                f"below '{step_3_dir}'."
            )
        with open(label_paths[0], encoding="utf-8") as label_file:
            io_set = set(map(str, json.load(label_file)))
        write_report(
            f"'{campaign_name}' has no embedded io_drive_label; "
            f"loaded labels from: {label_paths[0]}"
        )

    df_features = df_features.drop(
        columns=[
            "io_drive_label",
            "louvain_community",
            "top_topic",
            "boost_score",
        ],
        errors="ignore",
    ).select_dtypes(include="number")

    if feature_columns is not None:
        missing_columns = sorted(set(feature_columns) - set(df_features.columns))
        if missing_columns:
            raise ValueError(
                f"'{campaign_name}' is missing shared features: {missing_columns}"
            )
        df_features = df_features[feature_columns].copy()

    if normalize:
        numeric_columns = df_features.columns
        column_min = df_features[numeric_columns].min()
        column_range = (
            df_features[numeric_columns].max() - column_min
        ).replace(0, 1)
        df_features[numeric_columns] = (
            df_features[numeric_columns] - column_min
        ) / column_range

    if prefix_ids:
        df_features.index = campaign_name + "_" + df_features.index.astype(str)
        io_set = {f"{campaign_name}_{author_id}" for author_id in io_set}

    df_features["campaign_name"] = campaign_name
    n_io_in_graph = len(set(df_features.index.astype(str)) & io_set)
    write_report(
        f"Loaded '{campaign_name}' ({path}) | {len(df_features):,} authors "
        f"| {len(io_set):,} IO authors (dataset) "
        f"| {n_io_in_graph:,} IO in graph"
    )
    return df_features, io_set


def load_multiple_campaigns(
    campaign_paths: dict[str, Path],
    feature_columns: list[str] | None = None,
) -> tuple[pd.DataFrame, set[str]]:
    """Load and concatenate multiple campaign feature tables.

    Args:
        campaign_paths: Campaign names mapped to indicator parquet paths.
        feature_columns: Optional ordered numeric feature list.

    Returns:
        Combined feature DataFrame and union of campaign-prefixed IO authors.
    """
    frames = []
    combined_io = set()
    for campaign_name, path in campaign_paths.items():
        campaign_df, io_set = load_country_data(
            campaign_name,
            path,
            prefix_ids=True,
            feature_columns=feature_columns,
        )
        frames.append(campaign_df)
        combined_io.update(io_set)
    return pd.concat(frames, axis=0), combined_io

In [ ]:
# Use the same feature schema for every train/test campaign combination.
def get_common_numeric_feature_columns(
    campaign_paths: dict[str, Path],
) -> list[str]:
    """Return numeric model features present in every campaign.

    Args:
        campaign_paths: Campaign names mapped to indicator parquet paths.

    Returns:
        Numeric feature names shared by every campaign, in source order.
    """
    excluded_columns = {
        "io_drive_label",
        "louvain_community",
        "top_topic",
        "boost_score",
    }
    numeric_columns_by_campaign = {}
    for campaign_name, path in campaign_paths.items():
        campaign_df = pd.read_parquet(path)
        numeric_columns_by_campaign[campaign_name] = [
            column
            for column in campaign_df.select_dtypes(include="number").columns
            if column not in excluded_columns
        ]

    first_campaign = next(iter(campaign_paths))
    common_columns = set.intersection(
        *(set(columns) for columns in numeric_columns_by_campaign.values())
    )
    ordered_columns = [
        column
        for column in numeric_columns_by_campaign[first_campaign]
        if column in common_columns
    ]
    if not ordered_columns:
        raise ValueError("The selected campaigns have no numeric features in common.")
    return ordered_columns


FEATURE_COLUMNS_USED = get_common_numeric_feature_columns(campaign_base_dirs)
write_report(
    f"Common feature columns used ({len(FEATURE_COLUMNS_USED)}): "
    f"{FEATURE_COLUMNS_USED}"
)

## Evaluation helpers

In [ ]:
SCORING = {
    "auc_roc": "roc_auc",
    "auc_pr": "average_precision",
    "f1": make_scorer(f1_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
}
METRIC_COLUMNS = list(SCORING)


def build_within_campaign_splits(
    labels: np.ndarray,
    train_fraction: float,
    n_splits: int = N_SPLITS,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Create repeated stratified within-campaign partitions.

    Args:
        labels: Binary campaign labels.
        train_fraction: Fraction of campaign accounts used for training.
        n_splits: Number of repeated partitions.

    Returns:
        Disjoint training and test indices for each partition.
    """
    splitter = StratifiedShuffleSplit(
        n_splits=n_splits, train_size=train_fraction, random_state=RANDOM_STATE,
    )
    splits = list(splitter.split(np.zeros(len(labels)), labels))
    for train, test in splits:
        assert not np.intersect1d(train, test).size
        assert set(labels[train]) == set(labels[test]) == {0, 1}
        assert abs(len(train) / len(labels) - train_fraction) <= 1 / len(labels)
    return splits


def cross_validate_experiment(
    features: pd.DataFrame,
    labels: np.ndarray,
    cv_splits: list[tuple[np.ndarray, np.ndarray]],
    context: dict,
) -> pd.DataFrame:
    """Evaluate one experiment and return one row per split.

    Args:
        features: Numeric model features.
        labels: Binary labels aligned with features.
        cv_splits: Explicit train/test index pairs.
        context: Metadata shared by the returned rows.

    Returns:
        Split-level scores, ROC coordinates, and metadata.
    """
    if "campaign_name" in features:
        raise ValueError("campaign_name must not be a model feature")

    collect_roc = np.isclose(
        context["train_fraction"],
        ROC_TRAIN_FRACTION,
    )
    scores = cross_validate(
        get_classifier(context["model"]),
        features,
        labels,
        cv=cv_splits,
        scoring=SCORING,
        n_jobs=-1,
        error_score="raise",
        return_estimator=collect_roc,
    )
    result_rows = []
    for split_index, (train, test) in enumerate(cv_splits):
        false_positive_rate = None
        true_positive_rate = None
        if collect_roc:
            estimator = scores["estimator"][split_index]
            positive_class_index = np.flatnonzero(
                estimator.classes_ == 1
            ).item()
            positive_scores = estimator.predict_proba(features.iloc[test])[
                :, positive_class_index
            ]
            false_positive_rate, true_positive_rate, _ = roc_curve(
                labels[test],
                positive_scores,
            )
        result_rows.append({
            **context,
            "split": split_index + 1,
            "train_size": len(train),
            "test_size": len(test),
            "random_state": RANDOM_STATE,
            "roc_fpr": (
                json.dumps(false_positive_rate.tolist())
                if collect_roc
                else None
            ),
            "roc_tpr": (
                json.dumps(true_positive_rate.tolist())
                if collect_roc
                else None
            ),
            **{
                metric: float(scores[f"test_{metric}"][split_index])
                for metric in METRIC_COLUMNS
            },
        })
    return pd.DataFrame(result_rows)

In [ ]:
def run_experiment_sweep(
    campaigns: list[str],
    fractions: list[float],
    models: list[str],
    n_splits: int = N_SPLITS,
) -> pd.DataFrame:
    """Evaluate within-campaign classification across training fractions.

    Args:
        campaigns: Target campaigns to evaluate.
        fractions: Labeled training fractions for each campaign.
        models: Classifier names accepted by get_classifier.
        n_splits: Repeated stratified partitions per fraction.

    Returns:
        One result row per campaign, fraction, model, and split.
    """
    result_frames = []
    for campaign in tqdm(campaigns, desc="Within-campaign evaluation"):
        campaign_features, io_authors = load_country_data(
            campaign, campaign_base_dirs[campaign], feature_columns=FEATURE_COLUMNS_USED,
        )
        labels = np.asarray(campaign_features.index.astype(str).isin(io_authors), dtype=np.int8)
        features = campaign_features[FEATURE_COLUMNS_USED].reset_index(drop=True)
        for fraction in fractions:
            splits = build_within_campaign_splits(labels, fraction, n_splits)
            for model_name in models:
                write_report(
                    f"{campaign} | {model_name} | {fraction:.0%} training | {n_splits} splits",
                    transductive_output_path,
                )
                result_frames.append(cross_validate_experiment(
                    features, labels, splits,
                    {
                        "training_campaign": campaign,
                        "target_campaign": campaign,
                        "graph_filtering_setting": graph_filtering_setting,
                        "train_fraction": fraction,
                        "model": model_name,
                        "experiment_type": "transductive",
                    },
                ))
    results = pd.concat(result_frames, ignore_index=True)
    expected = len(campaigns) * len(fractions) * len(models) * n_splits
    if len(results) != expected:
        raise RuntimeError(f"Expected {expected:,} rows, got {len(results):,}")
    return results

In [ ]:
LEAVE_ONE_OUT_SCORING = {
    "auc_roc": "roc_auc",
    "auc_pr": "average_precision",
}


def build_leave_one_campaign_out_splits(
    features: pd.DataFrame,
    labels: np.ndarray,
    campaign_groups: np.ndarray,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Create one stratified group fold per campaign.

    Args:
        features: Model features aligned with labels and campaign groups.
        labels: Binary IO-author labels.
        campaign_groups: Campaign name for every account.

    Returns:
        Train/test index pairs, with one complete campaign in each test set.

    Raises:
        ValueError: If fewer than two campaigns are available, a fold does
            not hold out exactly one campaign, or a fold lacks either class.
    """
    unique_campaigns = np.unique(campaign_groups)
    if len(unique_campaigns) < 2:
        raise ValueError("At least two campaigns are required.")

    splitter = StratifiedGroupKFold(
        n_splits=len(unique_campaigns),
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    splits = list(splitter.split(features, labels, groups=campaign_groups))

    held_out_campaigns = []
    for train_indices, test_indices in splits:
        train_campaigns = set(campaign_groups[train_indices])
        test_campaigns = set(campaign_groups[test_indices])
        if len(test_campaigns) != 1 or train_campaigns & test_campaigns:
            raise ValueError(
                "Each fold must hold out exactly one complete campaign."
            )
        if set(labels[train_indices]) != {0, 1}:
            raise ValueError("A training fold contains only one class.")
        if set(labels[test_indices]) != {0, 1}:
            raise ValueError("A target campaign contains only one class.")
        held_out_campaigns.extend(test_campaigns)

    if set(held_out_campaigns) != set(unique_campaigns):
        raise ValueError("Not every campaign was held out exactly once.")
    return splits


def run_leave_one_campaign_out(
    campaigns: list[str],
    models: list[str],
) -> pd.DataFrame:
    """Train on all non-target campaigns and evaluate on each target.

    Args:
        campaigns: Campaigns to use as the cross-validation groups.
        models: Classifier names accepted by ``get_classifier``.

    Returns:
        One row per held-out campaign and model with ROC AUC and PR AUC.
    """
    selected_paths = {
        campaign: campaign_base_dirs[campaign] for campaign in campaigns
    }
    all_features, io_authors = load_multiple_campaigns(
        selected_paths,
        feature_columns=FEATURE_COLUMNS_USED,
    )
    features = all_features[FEATURE_COLUMNS_USED].reset_index(drop=True)
    labels = all_features.index.astype(str).isin(io_authors).astype(np.int8)
    labels = np.asarray(labels, dtype=np.int8)
    campaign_groups = all_features["campaign_name"].to_numpy()
    cv_splits = build_leave_one_campaign_out_splits(
        features,
        labels,
        campaign_groups,
    )

    result_rows = []
    for model_name in tqdm(models, desc="Leave-one-campaign-out models"):
        scores = cross_validate(
            get_classifier(model_name),
            features,
            labels,
            cv=cv_splits,
            scoring=LEAVE_ONE_OUT_SCORING,
            n_jobs=-1,
            error_score="raise",
        )
        for fold_number, (train_indices, test_indices) in enumerate(
            cv_splits, start=1
        ):
            target_campaign = np.unique(campaign_groups[test_indices]).item()
            training_campaigns = sorted(
                np.unique(campaign_groups[train_indices]).tolist()
            )
            result_rows.append({
                "fold": fold_number,
                "target_campaign": target_campaign,
                "training_campaigns": ",".join(training_campaigns),
                "experiment_type": "leave_one_campaign_out",
                "model": model_name,
                "train_size": len(train_indices),
                "test_size": len(test_indices),
                "train_positive_rate": labels[train_indices].mean(),
                "test_positive_rate": labels[test_indices].mean(),
                "auc_roc": float(scores["test_auc_roc"][fold_number - 1]),
                "auc_pr": float(scores["test_auc_pr"][fold_number - 1]),
                "graph_filtering_setting": graph_filtering_setting,
                "random_state": RANDOM_STATE,
            })
    return pd.DataFrame(result_rows)

## Plotting helpers

In [ ]:
MODEL_STYLES = {
    "xgboost": ("XGBoost", "-", "o"),
    "random_forest": ("Random Forest", "--", "s"),
    "logistic_regression": ("Logistic Regression", ":", "^"),
}
CAMPAIGN_ORDER = list(campaign_base_dirs)
campaign_palette = plt.get_cmap("tab10")
CAMPAIGN_COLORS = {
    campaign: campaign_palette(index % 10)
    for index, campaign in enumerate(CAMPAIGN_ORDER)
}
METRIC_LABELS = {
    "auc_roc": "ROC AUC",
    "auc_pr": "PR AUC",
    "f1": "F1",
    "recall": "Recall",
    "precision": "Precision",
}

In [ ]:
def plot_campaign_performance(
    results: pd.DataFrame,
    metric: str,
    experiment_type: str,
    model_name: str,
    output_dir: Path | None = None,
) -> None:
    """Plot campaign learning curves and mean ROC curves at the selected training fraction.

    Args:
        results: Split-level experiment results.
        metric: Metric to plot.
        experiment_type: Within-campaign learning setting (transductive).
        model_name: Classifier to include in the figure.
        output_dir: Optional figure directory.
    """
    plot_data = results[
        (results["experiment_type"] == experiment_type)
        & (results["model"] == model_name)
    ]
    represented_campaigns = set(plot_data["target_campaign"])
    campaigns = [
        campaign
        for campaign in CAMPAIGN_ORDER
        if campaign in represented_campaigns
    ]
    if not campaigns:
        write_report(
            f"No data to plot for '{model_name}', '{experiment_type}', "
            f"and '{metric}'."
        )
        return

    required_roc_columns = {"roc_fpr", "roc_tpr"}
    missing_roc_columns = required_roc_columns - set(plot_data.columns)
    if missing_roc_columns:
        raise ValueError(
            "ROC coordinates are missing from the results. Rerun the "
            "experiment sweep before creating this figure."
        )

    figure, (curve_axis, roc_axis) = plt.subplots(
        1,
        2,
        figsize=(28, 14),
        dpi=150,
    )
    for campaign in campaigns:
        campaign_data = plot_data[plot_data["target_campaign"] == campaign]
        summary = (
            campaign_data.groupby("train_fraction")[metric]
            .agg(["mean", "std"])
            .reset_index()
        )
        if summary.empty:
            continue

        means = summary["mean"]
        std = summary["std"].fillna(0)
        error = np.vstack([
            means - np.maximum(means - std, 0),
            np.minimum(means + std, 1) - means,
        ])
        curve_axis.errorbar(
            summary["train_fraction"] * 100,
            means,
            yerr=error,
            color=CAMPAIGN_COLORS[campaign],
            linestyle="-",
            marker="o",
            capsize=3,
            linewidth=2,
            markersize=6,
        )

    learning_label = experiment_type.title()
    model_label = MODEL_STYLES.get(
        model_name,
        (model_name.replace("_", " ").title(), "-", "o"),
    )[0]
    title = (
        f"{model_label} - {METRIC_LABELS[metric]} for All Campaigns - "
        f"{learning_label} Learning"
    )
    filename = (
        f"experiment_results_{CONFIG_TAG}_{model_name}_{experiment_type}_{metric}"
        "_all_campaigns.png"
    )

    percentages = [fraction * 100 for fraction in FRACTIONS]
    y_minimum = 0.9 if metric == "auc_roc" else 0.3
    curve_axis.set(
        xticks=percentages,
        xticklabels=[f"{value:g}%" for value in percentages],
        ylim=(y_minimum, 1.01),
        xlabel="Labeled target training data",
        ylabel=METRIC_LABELS[metric],
    )
    curve_axis.tick_params(axis="both", labelsize=26)
    curve_axis.xaxis.label.set_size(34)
    curve_axis.yaxis.label.set_size(34)
    curve_axis.set_title(
        "Score by labeled training data",
        fontsize=32,
    )
    curve_axis.grid(True, linestyle="--", alpha=0.4)

    common_false_positive_rate = np.linspace(0, 1, 201)
    for campaign in campaigns:
        campaign_roc_data = plot_data.loc[
            plot_data["target_campaign"].eq(campaign)
            & np.isclose(
                plot_data["train_fraction"],
                ROC_TRAIN_FRACTION,
            )
        ]
        interpolated_true_positive_rates = []
        for _, split_row in campaign_roc_data.iterrows():
            false_positive_rate = np.asarray(
                json.loads(split_row["roc_fpr"]),
                dtype=float,
            )
            true_positive_rate = np.asarray(
                json.loads(split_row["roc_tpr"]),
                dtype=float,
            )
            interpolated = np.interp(
                common_false_positive_rate,
                false_positive_rate,
                true_positive_rate,
            )
            interpolated[0] = 0
            interpolated_true_positive_rates.append(interpolated)
        if not interpolated_true_positive_rates:
            continue

        mean_true_positive_rate = np.mean(
            interpolated_true_positive_rates,
            axis=0,
        )
        mean_true_positive_rate[-1] = 1
        mean_auc = campaign_roc_data["auc_roc"].mean()
        roc_axis.plot(
            common_false_positive_rate,
            mean_true_positive_rate,
            color=CAMPAIGN_COLORS[campaign],
            linewidth=3,
            label=f"{campaign.replace('_', ' ')} (AUC = {mean_auc:.3f})",
        )

    roc_axis.plot(
        [0, 1],
        [0, 1],
        color="black",
        linestyle="--",
        linewidth=2,
        label="Random classifier",
    )
    roc_axis.set(
        xlim=(-0.01, 1.01),
        ylim=(-0.01, 1.01),
        xlabel="False positive rate",
        ylabel="True positive rate",
    )
    roc_axis.tick_params(axis="both", labelsize=26)
    roc_axis.xaxis.label.set_size(34)
    roc_axis.yaxis.label.set_size(34)
    roc_axis.set_title(
        f"ROC curves at {ROC_TRAIN_FRACTION:.0%} labeled training data",
        fontsize=32,
    )
    roc_axis.grid(True, linestyle="--", alpha=0.4)
    roc_axis.legend(fontsize=24, loc="lower right")

    campaign_handles = [
        Line2D([0], [0], color=CAMPAIGN_COLORS[campaign], linewidth=3, label=campaign)
        for campaign in campaigns
    ]
    curve_axis.legend(
        handles=campaign_handles,
        fontsize=26,
        loc="lower right",
        ncol=2,
    )

    figure.suptitle(title, fontsize=21, fontweight="bold")
    figure.text(
        0.5,
        0,
        "Transductive IO classification: Classify IO accounts by learning patterns "
        "from the same campaign/network."
        "\n"
        f"Points show mean +/- one standard deviation, and ROC curves show the mean across {N_SPLITS} stratified splits.",
        ha="center",
        va="bottom",
        fontsize=18,
    )
    figure.tight_layout(rect=[0, 0.15, 1, 0.93], w_pad=3.0)
    if output_dir is not None:
        plot_path = Path(output_dir) / filename
        figure.savefig(plot_path, dpi=600, bbox_inches="tight")
        write_report(f"Saved plot to: {plot_path}")
    plt.show()
    plt.close(figure)

# Transductive Within-Campaign

## Aggregate saved classification sweeps

Compare configurations available across the selected campaigns at the selected top percentage. Preserve the original aggregation rules and report metric availability before averaging.

In [ ]:
def get_results_path(project_root, campaign):
    """Build the classification sweep results path for a campaign.

    Args:
        project_root: Root directory of the project.
        campaign: Campaign directory name.

    Returns:
        Path to the campaign's classification sweep CSV file.
    """
    return (
        project_root
        / "results"
        / "Labeled_Datasets"
        / campaign
        / f"{campaign}_part_all"
        / "step_3_key_authors_graph"
        / "classification_sweep"
        / "classification_sweep_results.csv"
    )

### Load campaign sweeps

In [ ]:
if not CAMPAIGNS:
    raise ValueError("CAMPAIGNS must contain at least one campaign name.")

campaign_results = {}

for campaign in CAMPAIGNS:
    results_path = get_results_path(project_root, campaign)
    campaign_df = pd.read_csv(results_path)
    campaign_df = campaign_df.loc[
        campaign_df["top_percentage"].eq(top_percentage)
    ]
    campaign_df["indicators_configuration"] = campaign_df[
        "indicators_configuration"
    ].str.replace(
        " (Unweighted Clustering, No Square Clustering)",
        "",
        regex=False,
    )
    campaign_results[campaign] = campaign_df

### Configuration availability

In [ ]:
configuration_columns = [
    "topic_col",
    "account_relations_configuration",
    "indicators_configuration",
]
configuration_presence_records = []
common_configuration_values = {}

for configuration_column in configuration_columns:
    values_by_campaign = {
        campaign: set(campaign_df[configuration_column].dropna().unique())
        for campaign, campaign_df in campaign_results.items()
    }
    common_configuration_values[configuration_column] = set.intersection(
        *values_by_campaign.values()
    )
    all_configuration_values = set.union(*values_by_campaign.values())

    for configuration_value in sorted(all_configuration_values):
        present_in = [
            campaign
            for campaign, values in values_by_campaign.items()
            if configuration_value in values
        ]
        missing_from = [
            campaign
            for campaign, values in values_by_campaign.items()
            if configuration_value not in values
        ]
        configuration_presence_records.append(
            {
                "configuration": configuration_column,
                "value": configuration_value,
                "present_in": ", ".join(present_in),
                "missing_from": ", ".join(missing_from) or "None",
                "present_in_all_datasets": not missing_from,
            }
        )

configuration_presence_df = pd.DataFrame(configuration_presence_records)
print("Configuration availability by campaign:")
display(configuration_presence_df)

### Metric availability

In [ ]:
requested_summary_metric_columns = {
    "all_authors_auc": "ROC_AUC",
    "all_authors_auc_pr": "PR_AUC",
}
metric_presence_records = []
summary_metric_columns = {}
for metric_column, metric_name in requested_summary_metric_columns.items():
    present_in = [
        campaign
        for campaign, campaign_df in campaign_results.items()
        if metric_column in campaign_df
        and campaign_df[metric_column].notna().any()
    ]
    missing_from = [
        campaign for campaign in CAMPAIGNS if campaign not in present_in
    ]
    metric_presence_records.append(
        {
            "metric": metric_name,
            "source_column": metric_column,
            "present_in": ", ".join(present_in),
            "missing_from": ", ".join(missing_from) or "None",
            "present_in_all_datasets": not missing_from,
        }
    )
    if not missing_from:
        summary_metric_columns[metric_column] = metric_name

metric_presence_df = pd.DataFrame(metric_presence_records)
print("Metric availability by campaign:")
display(metric_presence_df)

### Detailed averages

In [ ]:
all_results_df = pd.concat(campaign_results.values(), ignore_index=True)
for configuration_column, common_values in common_configuration_values.items():
    all_results_df = all_results_df.loc[
        all_results_df[configuration_column].isin(common_values)
    ]
all_results_df = all_results_df.reset_index(drop=True)
group_columns = all_results_df.select_dtypes(exclude="number").columns.tolist()
aggregated_df = (
    all_results_df.groupby(group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(group_columns)
    .reset_index(drop=True)
)

display(aggregated_df)

classification_group_columns = configuration_columns
classification_metric_columns = [
    column
    for column in all_results_df.columns
    if column.startswith(("graph_authors_", "all_authors_"))
    and all(
        column in campaign_df and campaign_df[column].notna().any()
        for campaign_df in campaign_results.values()
    )
]
classification_metrics_df = (
    all_results_df[
        classification_group_columns + classification_metric_columns
    ]
    .groupby(classification_group_columns, as_index=False, dropna=False)
    .mean(numeric_only=True)
    .sort_values(classification_group_columns)
    .reset_index(drop=True)
)

display(classification_metrics_df)

### Configuration summaries

In [ ]:
topic_col_aggregation_df = (
    all_results_df.groupby("topic_col", as_index=False, dropna=False)[
        list(summary_metric_columns)
    ]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("topic_col")
    .reset_index(drop=True)
)
display(topic_col_aggregation_df)

account_relations_aggregation_df = (
    all_results_df.groupby(
        "account_relations_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns)]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("account_relations_configuration")
    .reset_index(drop=True)
)
display(account_relations_aggregation_df)

indicators_aggregation_df = (
    all_results_df.groupby(
        "indicators_configuration", as_index=False, dropna=False
    )[list(summary_metric_columns) + ["indicators_runtime_seconds"]]
    .mean()
    .rename(columns=summary_metric_columns)
    .sort_values("indicators_configuration")
    .reset_index(drop=True)
)
display(indicators_aggregation_df)

In [ ]:
summary_tables = {
    "configuration_availability": configuration_presence_df,
    "metric_availability": metric_presence_df,
    "aggregated_classification_sweep_results": aggregated_df,
    "aggregated_classification_metrics": classification_metrics_df,
    "aggregated_classification_sweep_results_by_topic_col": topic_col_aggregation_df,
    "aggregated_classification_sweep_results_by_account_relations_configuration": account_relations_aggregation_df,
    "aggregated_classification_sweep_results_by_indicators_configuration": indicators_aggregation_df,
}
for name, table in summary_tables.items():
    save_summary(table, summary_output_path, name)

## Filtered model comparisons

In [ ]:
filtered_df = (
    aggregated_df.loc[
        aggregated_df["indicators_configuration"].eq(indicators_configuration)
        & aggregated_df["topic_col"].eq(topic_method),
        [
            "model",
            "account_relations_configuration",
            "all_authors_auc",
            "all_authors_auc_pr",
        ],
    ]
    .rename(
        columns={
            "all_authors_auc": "roc_auc",
            "all_authors_auc_pr": "pr_auc",
        }
    )
    .reset_index(drop=True)
)

filtered_df

In [ ]:
metric_columns = ["roc_auc", "pr_auc"]
average_rows = (
    filtered_df.groupby(
        "account_relations_configuration",
        as_index=False,
    )[metric_columns]
    .mean()
)
average_rows.insert(0, "model", "all models average")

filtered_results_df = pd.concat(
    [filtered_df, average_rows],
    ignore_index=True,
)

save_summary(filtered_results_df, summary_output_path, f"filtered_aggregated_classification_results_{topic_method}_{indicators_configuration}")
display(filtered_results_df.round(3))

## Indicator information gain

In [ ]:
campaign_frames = []
for campaign in CAMPAIGNS:
    csv_path = (
        results_root
        / campaign
        / f"{campaign}_part_all"
        / "step_3_key_authors_graph"
        / f"topic_col_{topic_method}_top_percentage_{top_percentage:g}"
        / "node_indicators"
        / "node_indicators_information_gain.csv"
    )
    campaign_df = (
        pd.read_csv(csv_path, index_col=0)
        .rename_axis("indicator")
        .reset_index()
    )
    campaign_df["campaign"] = campaign
    campaign_frames.append(campaign_df)

information_gain_campaigns_df = pd.concat(campaign_frames, ignore_index=True)
information_gain_campaigns_df

In [ ]:
information_gain_summary_df = (
    information_gain_campaigns_df.groupby("indicator", as_index=False)["information_gain"]
    .mean()
    .rename(columns={"information_gain": "mean_information_gain"})
    .sort_values("mean_information_gain", ascending=False)
    .reset_index(drop=True)
)

save_summary(information_gain_summary_df, summary_output_path, f"aggregated_node_indicators_information_gain_{topic_method}")
display(information_gain_summary_df.round(3))

## Within-campaign classification

For each campaign and training fraction, evaluate repeated stratified partitions using only that campaign. Models share the same partitions.

In [ ]:
write_report("Starting within-campaign IO classification sweep.")
write_report(f"Campaigns: {CAMPAIGNS}")
write_report(f"Target training fractions: {FRACTIONS}")
write_report(f"Models: {MODELS}")
write_report(f"Stratified splits: {N_SPLITS}")
write_report(f"Random state: {RANDOM_STATE}")
write_report(f"Graph filter: {graph_filtering_setting}")
write_report(f"Key-author selection percentage: {top_percentage:g}%")
write_report(f"Output dir: {transductive_output_path}")

results_df = run_experiment_sweep(
    campaigns=CAMPAIGNS,
    fractions=FRACTIONS,
    models=MODELS,
    n_splits=N_SPLITS,
)

if results_df.empty:
    write_report(
        "WARNING: No results collected. Ensure every campaign has completed step 3."
    )
else:
    write_report(f"Sweep complete - {len(results_df):,} result rows collected.")
    display(results_df.head(12))

## Classification results

In [ ]:
if not results_df.empty:
    csv_path = transductive_output_path / f"experiment_results_{CONFIG_TAG}.csv"
    results_df.to_csv(csv_path, index=False)
    write_report(f"Split-level results saved to: {csv_path}")
    display(results_df.describe())

    agg_path = transductive_output_path / f"experiment_results_{CONFIG_TAG}_aggregated.csv"
    results_agg = (
        results_df
        .groupby(
            ["target_campaign", "experiment_type", "model", "train_fraction"]
        )
        .agg(
            auc_roc_mean=("auc_roc", "mean"),
            auc_roc_std=("auc_roc", "std"),
            auc_pr_mean=("auc_pr", "mean"),
            auc_pr_std=("auc_pr", "std"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", "std"),
            precision_mean=("precision", "mean"),
            precision_std=("precision", "std"),
            train_size_mean=("train_size", "mean"),
            test_size_mean=("test_size", "mean"),
            n_splits=("split", "nunique"),
        )
        .reset_index()
    )
    results_agg.to_csv(agg_path, index=False)
    write_report(f"Split-aggregated results saved to: {agg_path}")
    display(results_agg.head(12))
if not results_df.empty:
    results_agg.to_latex(transductive_output_path / f"experiment_results_{CONFIG_TAG}_aggregated.tex", index=False, float_format="%.3f")

## Learning and ROC curves

Produce ROC AUC and PR AUC learning-curve figures for each model, with ROC curves at the configured training fraction.

In [ ]:
if not results_df.empty:
    for model_name in MODELS:
        for metric in ("auc_roc", "auc_pr"):
            plot_campaign_performance(
                results_df, metric, "transductive", model_name,
                output_dir=transductive_output_path,
            )
write_report("Within-campaign evaluation completed.", transductive_output_path)

# Inductive Cross-Campaign

## Leave-one-campaign-out classification

Train on all accounts from every other campaign and test on the entire held-out campaign. Each campaign is held out once per model; no labeled target accounts are used for training. Training fractions do not apply here.

In [ ]:
write_report("Starting leave-one-campaign-out evaluation.", inductive_output_path)
leave_one_out_results_df = run_leave_one_campaign_out(CAMPAIGNS, MODELS)

campaign_order = {campaign: index for index, campaign in enumerate(CAMPAIGNS)}
leave_one_out_results_df["campaign_order"] = (
    leave_one_out_results_df["target_campaign"].map(campaign_order)
)
leave_one_out_results_df = (
    leave_one_out_results_df
    .sort_values(["campaign_order", "model"])
    .drop(columns="campaign_order")
    .reset_index(drop=True)
)

## Campaign comparisons and exports

In [ ]:
leave_one_out_path = (
    inductive_output_path / f"leave_one_campaign_out_results_{CONFIG_TAG}.csv"
)
leave_one_out_results_df.to_csv(leave_one_out_path, index=False)
write_report(f"Leave-one-campaign-out results saved to: {leave_one_out_path}")

leave_one_out_report = leave_one_out_results_df.pivot(
    index="target_campaign",
    columns="model",
    values=["auc_roc", "auc_pr"],
)
leave_one_out_report.columns = [
    f"{model}_{metric}" for metric, model in leave_one_out_report.columns
]
leave_one_out_report = leave_one_out_report.reindex(CAMPAIGNS)
leave_one_out_report.loc["Mean"] = leave_one_out_report.mean()
display(leave_one_out_report.style.format("{:.3f}"))
leave_one_out_report.to_latex(
    inductive_output_path / f"leave_one_campaign_out_results_{CONFIG_TAG}.tex",
    float_format="%.3f",
    caption=(
        "Leave-one-campaign-out evaluation of inductive IO classification. "
        "Each row shows the ROC AUC and PR AUC for a target campaign when "
        "training on all other campaigns. The final row shows the mean across "
        "all campaigns."
    )) 
print(f"Saved leave-one-campaign-out LaTeX table to: {inductive_output_path / f'leave_one_campaign_out_results_{CONFIG_TAG}.tex'}")
leave_one_out_report.to_csv(inductive_output_path / f"leave_one_campaign_out_summary_{CONFIG_TAG}.csv")
write_report("Pipeline completed.", inductive_output_path)